# Notebook 03: Monte Carlo Robustness Sweep

**Authors:** Neha Abin, Sahil Shah, Yajat Parmar  
**Affiliation:** Allen High School, Allen TX

---

This notebook runs a **100-trial Monte Carlo simulation** comparing three beamforming strategies:

1. **Always-ZF** — Zero-Forcing at all times
2. **Always-MRT** — Maximum Ratio Transmission at all times
3. **WaveLynk** — CCI-based adaptive switching

For each trial, we randomize velocity, SNR, channel seed, and delay, then measure:
- Outage rate (fraction of timesteps where SINR < threshold)
- Mean SINR
- Peak precoding error
- Number of mode switches (WaveLynk only)

Results are saved to `../data/simulation/monte_carlo_results.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.cci import compute_cci, DEFAULT_GAMMA, doppler_frequency, coherence_time
from src.channel_model import ChannelConfig, simulate_channel_sequence, add_csi_error
from src.beamforming import (
    zf_precoder, mrt_precoder, precoding_error, compute_sinr, normalize_precoder
)
from src.switching import WaveLynkController, SwitchingConfig

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12})

COLOR_ZF = '#ef4444'
COLOR_MRT = '#3b82f6'
COLOR_WAVELYNK = '#22c55e'

os.makedirs('../assets/notebooks', exist_ok=True)
os.makedirs('../data/simulation', exist_ok=True)

print('Setup complete.')

## Simulation Parameters

Each trial randomly samples from the following ranges:
- Velocity: 0.5–40 m/s (pedestrian to vehicular)
- SNR: 5–30 dB
- Feedback delay: 1–8 ms
- Channel seed: unique per trial

In [ ]:
# Simulation configuration
N_TRIALS = 100
fc = 6e9                    # Wi-Fi 7 6 GHz
Nr, Nt = 4, 4              # 4×4 MIMO
N_paths = 8                # multipath components
sim_duration = 0.05        # 50 ms per trial
dt = 0.001                 # 1 ms timestep
SINR_OUTAGE_THRESHOLD = 5  # 5 dB — below this counts as outage

# Randomized parameters
rng = np.random.default_rng(2025)
trial_velocities = rng.uniform(0.5, 40, N_TRIALS)
trial_snrs = rng.uniform(5, 30, N_TRIALS)
trial_delays = rng.uniform(0.001, 0.008, N_TRIALS)  # 1-8 ms
trial_seeds = rng.integers(0, 100000, N_TRIALS)

print(f'Running {N_TRIALS} trials...')
print(f'  Velocity range: {trial_velocities.min():.1f}–{trial_velocities.max():.1f} m/s')
print(f'  SNR range: {trial_snrs.min():.1f}–{trial_snrs.max():.1f} dB')
print(f'  Delay range: {trial_delays.min()*1000:.1f}–{trial_delays.max()*1000:.1f} ms')

## Run Monte Carlo Simulation

For each trial, we simulate the channel sequence and evaluate all three strategies
using identical channel conditions for a fair comparison.

In [ ]:
results = []

for trial_idx in range(N_TRIALS):
    v = trial_velocities[trial_idx]
    snr = trial_snrs[trial_idx]
    tau = trial_delays[trial_idx]
    seed = int(trial_seeds[trial_idx])
    
    # Generate channel sequence
    config = ChannelConfig(
        Nr=Nr, Nt=Nt, N_paths=N_paths, carrier_freq_hz=fc,
        velocity_mps=v, snr_db=snr, seed=seed
    )
    times, H_seq = simulate_channel_sequence(config, duration_s=sim_duration, dt_s=dt)
    noise_power = 10**(-snr / 10)
    delay_samples = max(1, int(tau / dt))
    
    # WaveLynk controller
    controller = WaveLynkController(SwitchingConfig(
        gamma=DEFAULT_GAMMA, carrier_freq_hz=fc
    ))
    
    # Per-timestep metrics
    sinr_zf_list, sinr_mrt_list, sinr_wl_list = [], [], []
    err_zf_list, err_mrt_list, err_wl_list = [], [], []
    
    for t_idx in range(delay_samples, len(times)):
        H_true = H_seq[t_idx]
        H_est = add_csi_error(H_true, snr, delay_samples, H_seq, t_idx)
        
        # ZF strategy
        W_zf_true = zf_precoder(H_true)
        W_zf_est = zf_precoder(H_est)
        W_zf_norm = normalize_precoder(W_zf_est)
        sinr_zf = 10 * np.log10(max(compute_sinr(H_true, W_zf_norm, noise_power), 1e-10))
        sinr_zf_list.append(sinr_zf)
        err_zf_list.append(precoding_error(W_zf_true, W_zf_est))
        
        # MRT strategy
        W_mrt_true = mrt_precoder(H_true)
        W_mrt_est = mrt_precoder(H_est)
        W_mrt_norm = normalize_precoder(W_mrt_est)
        sinr_mrt = 10 * np.log10(max(compute_sinr(H_true, W_mrt_norm, noise_power), 1e-10))
        sinr_mrt_list.append(sinr_mrt)
        err_mrt_list.append(precoding_error(W_mrt_true, W_mrt_est))
        
        # WaveLynk strategy
        W_wl, mode, cci = controller.select_precoder(H_est, v, tau, sinr_zf, dt)
        W_wl_norm = normalize_precoder(W_wl)
        sinr_wl = 10 * np.log10(max(compute_sinr(H_true, W_wl_norm, noise_power), 1e-10))
        sinr_wl_list.append(sinr_wl)
        
        if mode == 'ZF':
            err_wl_list.append(precoding_error(W_zf_true, W_wl))
        else:
            err_wl_list.append(precoding_error(W_mrt_true, W_wl))
    
    # Compute metrics
    outage_threshold_lin = SINR_OUTAGE_THRESHOLD
    
    for strategy, sinr_list, err_list, n_switches in [
        ('ZF', sinr_zf_list, err_zf_list, 0),
        ('MRT', sinr_mrt_list, err_mrt_list, 0),
        ('WaveLynk', sinr_wl_list, err_wl_list, len(controller.switch_events)),
    ]:
        sinr_arr = np.array(sinr_list)
        outage_rate = np.mean(sinr_arr < outage_threshold_lin)
        results.append({
            'trial': trial_idx + 1,
            'velocity_mps': v,
            'snr_db': snr,
            'strategy': strategy,
            'mean_sinr_db': float(np.mean(sinr_arr)),
            'outage_rate': float(outage_rate),
            'peak_error': float(np.max(err_list)),
            'num_switches': n_switches,
        })
    
    if (trial_idx + 1) % 20 == 0:
        print(f'  Completed {trial_idx + 1}/{N_TRIALS} trials')

df = pd.DataFrame(results)
print(f'\nDone! {len(df)} result rows generated.')

## Save Results to CSV

In [ ]:
csv_path = '../data/simulation/monte_carlo_results.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
print(f'Shape: {df.shape}')
df.head(9)

## Results Visualization

### Plot 1: SINR Distribution (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# --- Box plot: SINR distribution ---
ax = axes[0]
strategies = ['ZF', 'MRT', 'WaveLynk']
colors = [COLOR_ZF, COLOR_MRT, COLOR_WAVELYNK]

sinr_data = [df[df['strategy'] == s]['mean_sinr_db'].values for s in strategies]
bp = ax.boxplot(sinr_data, labels=strategies, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Mean SINR (dB)')
ax.set_title('SINR Distribution — 100 Trials')
ax.grid(True, alpha=0.3, axis='y')

# --- Bar chart: outage rate ---
ax = axes[1]
outage_rates = [df[df['strategy'] == s]['outage_rate'].mean() * 100 for s in strategies]
bars = ax.bar(strategies, outage_rates, color=colors, alpha=0.7, edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars, outage_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
           f'{rate:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Outage Rate (%)')
ax.set_title('Mean Outage Rate Comparison')
ax.grid(True, alpha=0.3, axis='y')

# --- Sorted per-trial outage rate ---
ax = axes[2]
for s, c in zip(strategies, colors):
    trial_outages = df[df['strategy'] == s]['outage_rate'].values
    sorted_outages = np.sort(trial_outages) * 100
    ax.plot(range(1, len(sorted_outages) + 1), sorted_outages,
           color=c, linewidth=2, label=s)
ax.set_xlabel('Trial (sorted by outage rate)')
ax.set_ylabel('Outage Rate (%)')
ax.set_title('Per-Trial Outage Rate (Tail Risk)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/notebooks/fig_08_monte_carlo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_08_monte_carlo.png')

## Summary Table

In [ ]:
summary = df.groupby('strategy').agg({
    'mean_sinr_db': ['mean', 'std', 'min'],
    'outage_rate': 'mean',
    'peak_error': ['mean', 'max'],
    'num_switches': 'mean',
}).round(3)

print('=' * 60)
print('MONTE CARLO RESULTS SUMMARY (100 Trials)')
print('=' * 60)
print()
print(summary.to_string())
print()
print('─' * 60)
print('Key Finding: WaveLynk combines the best of both strategies —')
print('ZF\'s high SINR in stable conditions + MRT\'s robustness near the cliff.')